# 1 · Build the corpus

For each chosen year: find every candidate paper (the facility's institutional repositories and
Europe PMC), fetch its full text, screen it with the rules, and embed it for the ranking.
Everything is stored in your working-data folder (in Google Drive on Colab) and is
**resumable**: an interrupted run continues where it stopped.

Run once per year (and again the following spring, when repositories have caught up). Then use
notebook **2 · Find and validate**.

**Time per year**: ~40–60 min to fetch and screen; embedding ~30–40 min on a CPU, a few minutes on
a GPU (Colab: *Runtime → Change runtime type → T4 GPU → Save*, **before** step 1).

**Run the cells one by one, from top to bottom** (▶ at the left of each cell, or Shift + Enter),
and wait for each to finish (a green tick in Colab).

> **Colab users**: nothing to install; step 2 connects your Google Drive, where all results are kept.
>
> **Local users** (JupyterLab): install once in a terminal, from a clone of the repository:
> `pip install -e ".[embed,openiris,notebook]" jupyterlab`, then `jupyter lab notebooks/`.
> Step 1 checks the installation and step 2 does nothing; choose a folder on your computer in step 3.
> (An installed app, e.g. built with LabConstrictor, needs no clone: step 1 downloads the latest
> rules and confirmed papers.)

## 1. Install

Downloads the code with the facility's rules and confirmed papers (Colab), or checks your local
installation. If Colab then shows **Restart session**, click it and run step 1 again.

In [ ]:
# @title 1. Install the code
REPO = 'https://github.com/AIC-Turku/facility-publication-finder'

import os, subprocess, sys
from pathlib import Path

if 'google.colab' in sys.modules:
    print('Google Colab: downloading and installing the code (about a minute) ...')
    CODE = Path('/content/code')          # temporary disk: fresh code, rules and confirmed papers every session
    if CODE.exists():
        subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only', '-q'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, str(CODE)], check=True)
    %pip install -q "{CODE}[embed,openiris]"
else:
    try:
        import facility_pubs
    except ImportError:
        raise SystemExit('Not installed: in a terminal, from the repository folder, run\n'
                         '  pip install -e ".[embed,openiris,notebook]" jupyterlab\n'
                         'then restart this notebook.') from None
    CODE = Path(facility_pubs.__file__).resolve().parents[2]

try:
    import ipywidgets  # noqa: F401  (preinstalled in Colab)
except ImportError:
    raise SystemExit('ipywidgets is missing: pip install ipywidgets, then restart the notebook.') from None
from facility_pubs import cli, ui, workflow
FACILITIES = ui.facilities_folder(CODE, REPO)   # an installed app downloads the latest rules and papers
print('✅ Code ready:', CODE, '· facilities:', FACILITIES)

## 2. Connect Google Drive

Colab asks *Permit this notebook to access your Google Drive files?*: click **Connect to Google
Drive**, choose your work account, allow access (**Select all** if offered) and **Continue**.

In [ ]:
# @title 2. Connect your Google Drive (Colab)
if ui.in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive connected: it is under /content/drive/MyDrive')
else:
    print('Running locally: no Google Drive to connect. Choose a folder on this computer in step 3.')

## 3. Settings

Choose the **facility**, the **working-data folder** and the **years to build**, then click
**Use these settings**.

* The folder: click a folder in the list to open it, **..** to go up; or type a path. In Colab your
  Drive is `/content/drive/MyDrive`. To start, create a folder (e.g. `AIC-publications`) with
  **Create folder here**. Use the **same folder in both notebooks** and share it only inside the
  facility: it will hold full texts and e-mail addresses.
* Years: Ctrl-click (Cmd-click on a Mac) to choose several.

In [ ]:
# @title 3. Choose the settings
settings = ui.Settings(FACILITIES, kind='build')
settings.widget

## 4. Optional private inputs

Skip unless you have an **OpenIRIS admin export** (`.xlsx`) or a **staff list** (`staff.csv`, one
"Lastname, Firstname" per line). They only add a private score and never leave this computer.
To use them set `UPLOAD_PRIVATE = True` and run the cell (Colab asks for the files; locally, copy
them into the folder it prints first).

In [ ]:
# @title 4. Optional private inputs
UPLOAD_PRIVATE = False

import shutil
cfg = settings.require()
PRIVATE = cfg['private']                  # never in Drive or the repository
if UPLOAD_PRIVATE and ui.in_colab():
    from google.colab import files
    for name in files.upload():
        shutil.move(name, PRIVATE / name)
if UPLOAD_PRIVATE:
    for x in sorted(PRIVATE.iterdir()):
        if x.suffix == '.xlsx':
            cli.main(['import-openiris', '--xlsx', str(x)])
print('private folder:', PRIVATE, '·', sorted(p.name for p in PRIVATE.iterdir()) or 'empty')

## 5. Fetch, screen and embed

The long step: keep the tab open. If it stops (Colab disconnected), click **Reconnect** and run
steps 1, 2, 3 (click *Use these settings* again) and 5: work already done is skipped.
It is done when the last lines show the ranking of each year (`wrote …/embedding_ranked.csv`).

In [ ]:
# @title 5. Fetch, screen and embed
cfg = settings.require()
for y in cfg['years']:
    print(f'=================== {y}')
    cli.main(['sweep', '--year', str(y)])
cli.main(['embed', '--years', *map(str, workflow.swept_years())])   # every built year improves the ranking

Done. Open [notebook 2 · Find and validate](https://colab.research.google.com/github/AIC-Turku/facility-publication-finder/blob/main/notebooks/2_find_and_validate.ipynb) to review a year. Switching between a GPU and a
CPU runtime later is fine: the embeddings already made are reused.